<a href="https://colab.research.google.com/github/RagaSandhiya05/LLM-From-Scratch/blob/main/04_Next_Word_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NEXT WORD PREDICTION**

**Import Libraries**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
print("PyTorch Version : " , torch.__version__)

PyTorch Version :  2.11.0+cpu


**Training Text**

In [2]:
text = """
i love python
i love machine learning
i love artificial intelligence
python is powerful
machine learning is fun
artificial intelligence is amazing
"""
print(text)


i love python
i love machine learning
i love artificial intelligence
python is powerful
machine learning is fun
artificial intelligence is amazing



**Tokenization and Vocabulary**

In [3]:
words = text.lower().split()
vocab = sorted(set(words))
word_to_id = {word : i for i , word in enumerate(vocab)}
id_to_word = {i : word for word , i in word_to_id.items()}
print("Vocabulary : " , vocab)
print("Vocabulary Size : " , len(vocab))

Vocabulary :  ['amazing', 'artificial', 'fun', 'i', 'intelligence', 'is', 'learning', 'love', 'machine', 'powerful', 'python']
Vocabulary Size :  11


**Create Training Data**

In [5]:
X = []
Y = []
for i in range(len(words) - 1) :
  X.append(word_to_id[words[i]])
  Y.append(word_to_id[words[i + 1]])
X = torch.tensor(X)
Y = torch.tensor(Y)
print("Input IDs : " , X)
print("Target IDs : " , Y)

Input IDs :  tensor([ 3,  7, 10,  3,  7,  8,  6,  3,  7,  1,  4, 10,  5,  9,  8,  6,  5,  2,
         1,  4,  5])
Target IDs :  tensor([ 7, 10,  3,  7,  8,  6,  3,  7,  1,  4, 10,  5,  9,  8,  6,  5,  2,  1,
         4,  5,  0])


**Build the Model**

In [6]:
class NextWordModel(nn.Module):
    def __init__(self , vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size , 16)
        self.fc1 = nn.Linear(16 , 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32 , vocab_size)
    def forward(self , x):
        x = self.embedding(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

**Create Model**

In [7]:
model = NextWordModel(len(vocab))
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters() , lr = 0.01)

**Train the Model**

In [8]:
epochs = 500
for epoch in range(epochs) :
  optimizer.zero_grad()
  output = model(X)
  loss = loss_function(output , Y)
  loss.backward()
  optimizer.step()
  if epoch % 50 == 0 :
    print(f"Epoch {epoch : 3d} | Loss = {loss.item() : .4f}")

Epoch   0 | Loss =  2.4577
Epoch  50 | Loss =  0.5131
Epoch  100 | Loss =  0.5124
Epoch  150 | Loss =  0.5123
Epoch  200 | Loss =  0.5122
Epoch  250 | Loss =  0.5121
Epoch  300 | Loss =  0.5121
Epoch  350 | Loss =  0.5121
Epoch  400 | Loss =  0.5121
Epoch  450 | Loss =  0.5120


**Prediction Function**

In [11]:
def predict_next_word(word) :
  if word not in word_to_id :
    print("Word not found in vocabulary.")
    return
  x = torch.tensor([word_to_id[word]])
  model.eval()
  with torch.no_grad() :
      prediction = model(x)
      predicted_id = torch.argmax(prediction , dim = 1).item()
      print("Input Word : " , word)
      print("Predicted Next Word : " , id_to_word[predicted_id])

**Test the Model**

In [12]:
predict_next_word("i")
predict_next_word("love")
predict_next_word("machine")
predict_next_word("artificial")
predict_next_word("python")

Input Word :  i
Predicted Next Word :  love
Input Word :  love
Predicted Next Word :  python
Input Word :  machine
Predicted Next Word :  learning
Input Word :  artificial
Predicted Next Word :  intelligence
Input Word :  python
Predicted Next Word :  i


**Display Embeddings**

In [14]:
print("Embeddings Matrix Shape : ")
print(model.embedding.weight.shape)

print("\nEmbedding Weights : ")
print(model.embedding.weight)

Embeddings Matrix Shape : 
torch.Size([11, 16])

Embedding Weights : 
Parameter containing:
tensor([[-0.1690, -1.4260, -0.1780, -0.5772, -0.7423, -1.7826,  0.8143, -0.6056,
         -0.4272,  1.0411, -0.2240,  0.9111, -1.7900,  0.6510, -1.0264, -0.9840],
        [ 0.4015, -0.2440,  1.1385, -2.2804,  1.6027, -0.1209,  0.7867,  2.1467,
          0.2941,  0.7221, -1.5483, -1.1312,  1.3382,  0.7465,  0.6844,  0.5886],
        [ 2.4867, -1.0893,  0.2482,  0.7091, -1.9545, -1.9952, -2.0052, -1.4779,
          0.1996,  0.1445, -1.3905, -1.0693, -0.4669,  1.4097,  0.2645, -0.3191],
        [ 0.9983,  0.1185, -2.2634,  1.3290,  0.4810,  1.0088, -1.6718,  0.9440,
          1.1471, -1.2257, -1.3717,  0.3312, -0.4010,  1.6259,  0.2086,  0.9674],
        [ 0.4225,  1.1552,  0.8496, -2.4227, -1.5097, -0.7015, -0.6622, -0.7279,
          2.0605,  0.8113,  1.3166, -2.8926, -0.9390,  0.5773,  1.5520, -1.6611],
        [-1.5862,  0.9118, -0.4741,  2.8586, -0.3165, -0.2580, -1.6411, -0.4460,
          0.